In [1]:
try:
    from pyspark import SparkContext, SparkConf
    from pyspark.sql import SparkSession
except ImportError as e:
    printmd('<<<<<!!!!! Please restart your kernel after installing Apache Spark !!!!!>>>>>')

In [ ]:
sc = SparkContext.getOrCreate(SparkConf().setMaster("local[*]"))

spark = SparkSession \
    .builder \
    .getOrCreate()

# Упражнение

## Часть 1
Теперь давайте вычислим ковариацию и корреляцию самостоятельно, используя ApacheSpark

Сначала создадим два случайных RDD, которые вообще не должны коррелировать.


In [ ]:
import random
rddX = sc.parallelize(random.sample(list(range(100)),100))
rddY = sc.parallelize(random.sample(list(range(100)),100))

Теперь вычислим среднее, обратите внимание, что мы явно приводим знаменатель к float, чтобы получить float вместо int

In [ ]:
meanX = rddX.sum()/float(rddX.count())
meanY = rddY.sum()/float(rddY.count())
print (meanX)
print (meanY)

Теперь вычислим ковариацию

In [ ]:
rddXY = rddX.zip(rddY)
covXY = rddXY.map(lambda x_y : (x_y[0]-meanX)*(x_y[1]-meanY)).sum()/rddXY.count()
covXY

Ковариация не является нормализованной мерой. Поэтому мы используем её для вычисления корреляции. Но перед этим нам нужно вычислить индивидуальные стандартные отклонения

In [ ]:
from math import sqrt
n = rddXY.count()
sdX = sqrt(rddX.map(lambda x : pow(x-meanX,2)).sum()/n)
sdY = sqrt(rddY.map(lambda x : pow(x-meanY,2)).sum()/n)
print (sdX)
print (sdY)

Теперь вычислим корреляцию

In [ ]:
corrXY = covXY / (sdX * sdY)
corrXY

## Часть 2
Теперь мы хотим создать матрицу корреляции из четырех RDD, использованных в лекции

In [ ]:
from pyspark.mllib.stat import Statistics
import random
column1 = sc.parallelize(range(100))
column2 = sc.parallelize(range(100,200))
column3 = sc.parallelize(list(reversed(range(100))))
column4 = sc.parallelize(random.sample(range(100),100))
data = column1.zip(column2).zip(column3).zip(column4).map(lambda a_b_c_d : (a_b_c_d[0][0][0],a_b_c_d[0][0][1],a_b_c_d[0][1],a_b_c_d[1]) ).map(lambda a_b_c_d : [a_b_c_d[0],a_b_c_d[1],a_b_c_d[2],a_b_c_d[3]])
print(Statistics.corr(data))

Поздравляем, вы закончили упражнение 2